# 01 — Preprocesamiento
## TFM: Predicción del comportamiento electoral presidencial en Colombia a nivel municipal

Este notebook **orquesta** la construcción del dataset maestro llamando a las funciones
de `src/`, que contienen toda la lógica pesada (ver `src/preprocesamiento.py`,
`src/agregacion_2022.py`, `src/integrar_nbi.py`, `src/integrar_victimas.py`,
`src/verificar_calidad_y_finalizar.py`).

**Fase cerrada.** Las decisiones metodológicas de este notebook están documentadas y
justificadas en la bitácora técnica interna de la Fase 1 (no versionada en este repo).
Este notebook es la evidencia ejecutable de esa fase, no el lugar para tomar nuevas
decisiones de preprocesamiento.

**Flujo:**
1. Extracción del % de voto de izquierda por municipio (1998-2018)
2. Agregación de 2022 (nivel mesa → municipio) con crosswalk DIVIPOLA
3. Consolidación del panel completo 1998-2022
4. Variable lag (electoral y votos en blanco), con protección anti-leakage
5. Integración del NBI (2005 y 2018)
6. Integración de víctimas del conflicto armado (PER_OCU)
7. Enriquecimiento: región natural (DANE)
8. Verificación de calidad final y guardado del dataset maestro


## 0. Configuración inicial

Sin rutas locales hardcodeadas: `BASE_DIR` es la única variable a ajustar.

In [1]:
import sys
import pandas as pd

BASE_DIR = ".."  # raíz del repo (este notebook vive en notebooks/)
sys.path.insert(0, f"{BASE_DIR}/src")

from preprocesamiento import (
    construir_serie_izquierda,
    calcular_variable_lag,
    anadir_region_dane,
)
from agregacion_2022 import cargar_y_agregar_2022
from integrar_nbi import integrar_nbi
from integrar_victimas import integrar_victimas
from verificar_calidad_y_finalizar import (
    estandarizar_nombres,
    marcar_confiabilidad_electoral,
    marcar_valido_para_modelado,
    verificar_calidad,
)

pd.set_option("display.max_columns", 30)
print("Módulos cargados correctamente.")

Módulos cargados correctamente.


## 1. Extracción del % de izquierda por municipio (1998-2018)

Se identifica al candidato de la línea Polo Democrático → Colombia Humana → Pacto
Histórico en cada año por su `codigo_lista` (no por nombre, que no es estable entre
años). Se excluye el voto en el exterior (`coddpto=9`) y se corrige un error
tipográfico real detectado en el fichero oficial de 2010 (Medio Atrato, Chocó:
`27415` → `27425`).

In [2]:
RUTAS_ELECTORALES = {
    1998: f"{BASE_DIR}/datos/raw/1998_presidencia_primera_vuelta_dta_e2a3b897bf.csv",
    2002: f"{BASE_DIR}/datos/raw/2002_presidencia_dta_c5a0392d8f.csv",
    2006: f"{BASE_DIR}/datos/raw/2006_presidencia_dta_9eb2e9319c.csv",
    2010: f"{BASE_DIR}/datos/raw/2010_presidencia_primera_vuelta_dta_b3dfcb91d9.csv",
    2014: f"{BASE_DIR}/datos/raw/2014_presidencia_primera_vuelta_dta_5cf9e9fded.csv",
    2018: f"{BASE_DIR}/datos/raw/2018_presidencia_primera_vuelta_dta_a61a1a0633.csv",
}

serie_1998_2018 = construir_serie_izquierda(RUTAS_ELECTORALES)
serie_1998_2018["divipola"] = serie_1998_2018["divipola"].astype(str)
serie_1998_2018.head()

  1998: 1084 municipios procesados


  2002: 1115 municipios procesados
  2006: 1118 municipios procesados


  Corregido: 11 filas de 2010 con codmpio=27415 -> 27425
  2010: 1122 municipios procesados
  2014: 1122 municipios procesados


  2018: 1122 municipios procesados


,divipola,ano,coddpto,departamento,municipio,votos_validos,votos_izquierda,pct_izquierda,votos_blanco,votos_nulos,votos_totales_emitidos,pct_votos_blanco,num_candidatos
0,05001,1998,5,ANTIOQUIA,MEDELLIN,451340,<NA>,<NA>,4733.0,1957.0,458030.0,1.033338,13
1,05002,1998,5,ANTIOQUIA,ABEJORRAL,3718,<NA>,<NA>,26.0,44.0,3788.0,0.686378,13
2,05004,1998,5,ANTIOQUIA,ABRIAQUI,858,<NA>,<NA>,6.0,5.0,869.0,0.690449,13
3,05021,1998,5,ANTIOQUIA,ALEJANDRIA,1577,<NA>,<NA>,10.0,1.0,1588.0,0.629723,13
4,05030,1998,5,ANTIOQUIA,AMAGA,6005,<NA>,<NA>,88.0,63.0,6156.0,1.429500,13


In [3]:
# Validacion contra resultados historicos oficiales conocidos (ponderado nacional)
for ano in [2006, 2010, 2014, 2018]:
    sub = serie_1998_2018[serie_1998_2018["ano"] == ano]
    pct_nacional = sub["votos_izquierda"].sum() / sub["votos_validos"].sum() * 100
    print(f"{ano}: {pct_nacional:.2f}% (ponderado nacional)")

2006: 22.58% (ponderado nacional)
2010: 9.32% (ponderado nacional)
2014: 16.23% (ponderado nacional)
2018: 25.72% (ponderado nacional)


## 2. Agregación de 2022 (mesa → municipio)

Los ficheros de 2022 se publican a nivel de mesa y usan una codificación DEP/MUN
**interna de la Registraduría, distinta al DIVIPOLA oficial**. Se resuelve con un
crosswalk por nombre normalizado contra la referencia de 2018 (ver
`src/agregacion_2022.py`). El candidato de izquierda (Gustavo Petro, Pacto Histórico)
se identifica por el código de PARTIDO (`PAR='1235'`), estable entre ambas vueltas —
el código de candidato (`CAN`) cambia entre 1ª y 2ª vuelta.

**Nota:** los ficheros originales de mesa (>30MB) no se versionan en este repositorio
(ver README, sección Datos) — están en la carpeta de Google Drive enlazada ahí.

In [4]:
RUTA_2022_1V_MESA = f"{BASE_DIR}/datos/raw_no_versionado/MMV_NACIONAL_PRESIDENTE_2022_1v.csv"  # descargar de Drive
RUTA_REFERENCIA_2018 = RUTAS_ELECTORALES[2018]

serie_2022 = cargar_y_agregar_2022(RUTA_2022_1V_MESA, RUTA_REFERENCIA_2018)
print("Municipios sin DIVIPOLA asignado:", serie_2022["divipola"].isna().sum())

pct_nacional_2022 = serie_2022["votos_izquierda"].sum() / serie_2022["votos_validos"].sum() * 100
print(f"2022 (1a vuelta): {pct_nacional_2022:.2f}% (ponderado nacional, oficial ~40.32%)")

Municipios sin DIVIPOLA asignado: 0
2022 (1a vuelta): 41.19% (ponderado nacional, oficial ~40.32%)


## 3. Consolidación del panel completo (1998-2022)

In [5]:
COLUMNAS_COMUNES = [
    "divipola", "ano", "departamento", "municipio", "votos_validos", "votos_izquierda",
    "pct_izquierda", "votos_blanco", "votos_nulos", "votos_totales_emitidos",
    "pct_votos_blanco", "num_candidatos",
]
panel = pd.concat(
    [serie_1998_2018[COLUMNAS_COMUNES], serie_2022[COLUMNAS_COMUNES]],
    ignore_index=True,
)
print("Shape panel completo:", panel.shape)
panel.groupby("ano").size()

Shape panel completo: (7804, 12)


ano
1998    1084
2002    1115
2006    1118
2010    1122
2014    1122
2018    1122
2022    1121
dtype: int64

## 4. Variable lag (electoral y votos en blanco)

Calculada por merge explícito año→año anterior (nunca por `shift()` posicional), lo
que garantiza que el lag de un año objetivo **solo** puede venir del año inmediatamente
anterior. `pct_votos_blanco` del año en curso **no** se usa como predictor (se
determina en la misma urna que el resultado); se usa su lag. Los casos sin lag
disponible (municipios muy remotos o de creación reciente) se imputan con la media
departamental del año, marcados con una bandera explícita — nunca en silencio. El
peso muestral (`peso_muestral = min(votos_totales_emitidos/500, 1.0)`) protege contra
la sobre-influencia de los municipios más grandes en el modelado, sin excluir a los
más pequeños.

Ver `tests/test_lag_electoral.py` para el test de regresión que protege esta función
contra fugas de datos.

In [6]:
panel_con_lag = calcular_variable_lag(panel)
print("Shape:", panel_con_lag.shape)
print("Filas con lag_pct_izquierda imputado:", panel_con_lag["lag_pct_izquierda_imputado"].sum())
print("Filas con peso_muestral < 1.0:", (panel_con_lag["peso_muestral"] < 1.0).sum())

AVISO: 23 filas de 'lag_pct_izquierda' imputadas con la media departamental del año (marcadas en 'lag_pct_izquierda_imputado').
AVISO: 22 filas de 'lag_pct_votos_blanco' imputadas con la media departamental del año (marcadas en 'lag_pct_votos_blanco_imputado').
Shape: (5605, 17)
Filas con lag_pct_izquierda imputado: 23
Filas con peso_muestral < 1.0: 128


## 5. Integración del NBI (2005 y 2018)

NBI Censo 2005 para 1998/2002/2006/2010; NBI Censo 2018 para 2014/2018/2022. A
diferencia del caso de 2022, el código DIVIPOLA de ambos ficheros de NBI coincide
directamente con el oficial (verificado contra Vaupés, incluyendo los sufijos
"(ANM)"), así que no hace falta crosswalk por nombre aquí.

In [7]:
RUTA_NBI_2005 = f"{BASE_DIR}/datos/raw/NBI_total_dpto_30_Jun_2012.xls"
RUTA_NBI_2018 = f"{BASE_DIR}/datos/raw/CNPV-2018-NBI.xlsx"

panel_con_nbi = integrar_nbi(panel_con_lag, RUTA_NBI_2005, RUTA_NBI_2018)
print("Nulos en nbi_total:", panel_con_nbi["nbi_total"].isna().sum())

Nulos en nbi_total: 0


## 6. Integración de víctimas del conflicto armado (PER_OCU)

`PER_OCU` se incorpora del **mismo año electoral**, no rezagado — a diferencia del
voto en blanco, es una característica estructural del territorio (intensidad del
conflicto), un dato público disponible antes de la jornada electoral, no un resultado
de la votación en sí. La ausencia de una fila en `VICTIMAS_FILTRADO_V2.csv` para un
departamento-año se interpreta como 0 víctimas registradas ese año (no como dato
faltante) — verificado con el caso de San Andrés desde 2020.

In [8]:
RUTA_VICTIMAS = f"{BASE_DIR}/datos/procesados/VICTIMAS_FILTRADO_V2.csv"

panel_con_victimas = integrar_victimas(panel_con_nbi, RUTA_VICTIMAS)
print("Nulos en per_ocu:", panel_con_victimas["per_ocu"].isna().sum())

NOTA: 2 filas sin registro en VICTIMAS_FILTRADO_V2.csv - se interpretan como 0 victimas (ausencia de registro en la fuente UARIV para ese departamento-año), no como dato faltante:
     divipola   ano departamento
5569    88001  2022   SAN ANDRES
5570    88564  2022   SAN ANDRES
Nulos en per_ocu: 0


## 7. Enriquecimiento: región natural (DANE)

Columna descriptiva/de navegación (EDA, app) — **no** entra en `COLUMNAS_PREDICTORAS`
por defecto. Mapeo por código DIVIPOLA de departamento, nunca por nombre de texto
(misma razón que el resto del casado de fuentes: los nombres varían entre años, los
códigos no). San Andrés se clasifica como **Caribe** (convención mayoritaria,
verificada contra múltiples fuentes externas) — "Insular" existe conceptualmente
pero no tiene ningún código de departamento asignado en este dataset (Gorgona/Malpelo
no tienen DIVIPOLA municipal propio). Ver `tests/test_region_dane.py`.

In [9]:
panel_con_region = anadir_region_dane(panel_con_victimas)
panel_con_region["region_dane"].value_counts()

region_dane
Andina       3145
Caribe        982
Pacifica      889
Orinoquia     295
Amazonia      294
Name: count, dtype: int64

## 8. Verificación de calidad final y guardado del dataset maestro

In [10]:
panel_final = estandarizar_nombres(panel_con_region, RUTA_REFERENCIA_2018)
panel_final = marcar_confiabilidad_electoral(panel_final, umbral_votos=30)
panel_final = marcar_valido_para_modelado(panel_final)

verificar_calidad(panel_final)

21 filas marcadas con baja_confiabilidad_electoral=1 (menos de 30 votos totales emitidos).
AVISO: 1 fila(s) marcada(s) como valido_para_modelado=0 (pct_izquierda indefinido, 0 votos validos ese año):
     divipola   ano departamento     municipio
1100    94885  2006      GUAINIA  LA GUADALUPE
VERIFICACION DE CALIDAD DEL DATASET MAESTRO

Filas totales: 5605
Duplicados (divipola+ano): 0

Filas por año:
ano
2006    1118
2010    1122
2014    1122
2018    1122
2022    1121
dtype: int64

Nulos por columna (excepto los ya tratados con bandera explicita):
pct_izquierda       1
pct_votos_blanco    1
dtype: int64

Rangos de variables clave:
  pct_izquierda: [0.00, 100.00]
  lag_pct_izquierda: [0.00, 100.00]
  pct_votos_blanco: [0.00, 51.73]
  lag_pct_votos_blanco: [0.00, 100.00]
  nbi_total: [1.59, 100.00]
  per_ocu: [0, 64755]

Consistencia de nombres por codigo DIVIPOLA (tras estandarizar):
  Codigos con nombre inconsistente entre años: 0

Banderas de calidad:
  baja_confiabilidad_electoral=1:

In [11]:
RUTA_SALIDA = f"{BASE_DIR}/datos/procesados/dataset_maestro_electoral.csv"
panel_final.to_csv(RUTA_SALIDA, index=False)
print(f"Dataset maestro guardado: {panel_final.shape[0]} filas x {panel_final.shape[1]} columnas")
print(f"-> {RUTA_SALIDA}")

Dataset maestro guardado: 5605 filas x 23 columnas


-> ../datos/procesados/dataset_maestro_electoral.csv


## Resumen

El dataset maestro queda guardado en `datos/procesados/dataset_maestro_electoral.csv`,
listo para el EDA (Chat 4) y la modelización (Chat 5). Las columnas predictoras
válidas están en `COLUMNAS_PREDICTORAS` (`src/preprocesamiento.py`); todo lo demás es
descriptivo, de peso muestral o de control de calidad — ver
`COLUMNAS_DESCRIPTIVAS_NO_PREDICTORAS`.

Las decisiones metodológicas detrás de cada paso (por qué se excluye el voto exterior,
por qué el lag se calcula así, por qué PER_OCU no se rezaga, etc.) están documentadas
y justificadas en la bitácora técnica interna de esta fase.